## Imports

In [2]:
# Cell 1: Setup and Imports
import sys
import os
from pathlib import Path

# Get project root
project_root = Path(os.getcwd()).parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from langfuse import get_client
from data.evaluation_dataset import EVALUATION_DATASET
from utils.utils import load_config, validate_langfuse_connection, logger

In [3]:
config = load_config()  # Now it will find config/config.json from project root
langfuse = get_client()

if not validate_langfuse_connection(langfuse):
    raise RuntimeError("❌ Langfuse authentication failed")

2026-02-10 12:00:55,155 - utils.utils - INFO - Configuration loaded successfully
2026-02-10 12:00:55,900 - utils.utils - INFO - Langfuse authentication successful


## DataSet Checking

In [2]:

# Cell 2: Create/Get Dataset
DATASET_NAME = config['dataset']['name']

try:
    dataset = langfuse.get_dataset(DATASET_NAME)
    logger.info(f"ℹ️ Dataset '{DATASET_NAME}' already exists.")
except Exception:
    langfuse.create_dataset(
        name=DATASET_NAME,
        description="Multi-turn conversations for evaluation"
    )
    logger.info(f"✅ Created dataset '{DATASET_NAME}'.")
    dataset = langfuse.get_dataset(DATASET_NAME)

2026-02-10 11:46:27,018 - utils.utils - INFO - ✅ Created dataset 'multi-turn-conversations2'.


## Uploading DataSet

In [3]:
# Cell 3: Upload Dataset Items
logger.info("⬆️ Uploading dataset items...")
for i, item in enumerate(EVALUATION_DATASET):
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        input=item["input"]
    )
    logger.info(f"   Uploaded item {i + 1}/{len(EVALUATION_DATASET)}")

logger.info("\n🎉 Dataset upload completed successfully!")

2026-02-10 11:46:31,439 - utils.utils - INFO - ⬆️ Uploading dataset items...
2026-02-10 11:46:31,516 - utils.utils - INFO -    Uploaded item 1/1
2026-02-10 11:46:31,517 - utils.utils - INFO - 
🎉 Dataset upload completed successfully!


## Viewing the DataSet

In [4]:
# Cell 4: View Dataset
dataset = langfuse.get_dataset(DATASET_NAME)
print(f"Dataset: {dataset.name}")
print(f"Total items: {len(dataset.items)}")
for i, item in enumerate(dataset.items[:3]):
    print(f"\nItem {i+1}:")
    print(f"  Persona: {item.input['persona']}")
    print(f"  Scenario: {item.input['scenario']}")

Dataset: multi-turn-conversations2
Total items: 1

Item 1:
  Persona: A project manager tracking deadlines and needs precise date calculations
  Scenario: Working on a project that starts October 28, 2024, with a 90-day deadline


## Testing the imports

In [5]:
# Test import
from metrics.metric import EvaluationMetrics
from utils.utils import load_config
import inspect

# Verify signature
sig = inspect.signature(EvaluationMetrics.__init__)
print("✅ Updated signature:")
print(sig)

# Test initialization
try:
    evaluator = EvaluationMetrics(
        judge_model=config['models']['judge_model'],
        ollama_config=config['ollama'],
        metrics_config=config['evaluation_metrics']
    )
    print("\n✅ Evaluator initialized successfully!")
    print(f"   Metrics: {list(evaluator.metrics_config.keys())}")
except Exception as e:
    print(f"\n❌ Error: {e}")

✅ Updated signature:
(self, judge_model: str, ollama_config: Dict, metrics_config: Dict)


2026-02-10 12:02:52,663 - utils.utils - INFO - Initialized EvaluationMetrics with 4 metrics



✅ Evaluator initialized successfully!
   Metrics: ['relevance', 'accuracy', 'helpfulness', 'clarity']


## Running the TASK

In [ ]:
import sys
import os
from pathlib import Path

# Navigate up from notebooks/ to project root
project_root = Path.cwd().parent  # Go up one level from notebooks/
sys.path.insert(0, str(project_root))

# Change working directory to project root
os.chdir(project_root)

# Now imports will work
from metrics.metric import EvaluationMetrics
from main import run_task
from utils.utils import load_config
from langfuse import get_client

# Load config (now relative to project root)
config = load_config("config/config.json")
langfuse = get_client()

# Get dataset
dataset = langfuse.get_dataset(config['dataset']['name'])

# Initialize evaluator
evaluator = EvaluationMetrics(
    judge_model=config['models']['judge_model'],
    ollama_config=config['ollama'],
    metrics_config=config['evaluation_metrics']
)

evaluators = evaluator.get_all_evaluators()
run_evaluators = [evaluator.create_average_quality_run_evaluator()]

result = dataset.run_experiment(
    name="notebook-experiment-v1",
    description="Multi-turn conversation experiment with automatic evaluation",
    task=run_task,
    evaluators=evaluators,
    run_evaluators=run_evaluators
)

langfuse.flush()
print("✅ Experiment complete!")

2026-02-10 14:08:16,734 - utils.utils - INFO - Configuration loaded successfully
2026-02-10 14:08:17,032 - utils.utils - INFO - Initialized EvaluationMetrics with 4 metrics
2026-02-10 14:08:17,114 - utils.utils - INFO - 
🔄 Starting continuous multi-turn conversation...
2026-02-10 14:08:17,623 - utils.utils - INFO - 
2026-02-10 14:08:17,623 - utils.utils - INFO - SESSION: session8-0753e157-0ea2-4a08-9477-8ab6082bec2a
2026-02-10 14:08:17,624 - utils.utils - INFO - SCENARIO: Working on a project that starts October 28, 2024, with a 90-day deadline
2026-02-10 14:08:17,624 - utils.utils - INFO - ================================================================================

2026-02-10 14:09:14,856 - utils.utils - INFO - --- Turn 1 ---
2026-02-10 14:09:14,857 - utils.utils - INFO - USER:  Hi there! I'm working on a project that is slated to start on October 28, 2024. The deadline for th...
2026-02-10 14:09:14,858 - utils.utils - INFO - ASSISTANT: To calculate the dates for each phase, let'